In [1]:
import pandas as pd
import numpy as np
import librosa
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load your saved training components
print("Loading saved training data...")
X_train, X_test, y_train, y_test = joblib.load("/kaggle/input/crimealert-ml/train_test_split.pkl")

# Load original CSV to get feature names
df = pd.read_csv("/kaggle/input/crimealert-analysis/audio_features.csv")
feature_names = df.drop(columns=["label", "file"]).columns.tolist()

# Recreate the scaler from training data
scaler = StandardScaler()
X_original = df[feature_names]
scaler.fit(X_original)

# Load your trained SVM model
svm_model = SVC(kernel="rbf", random_state=42, probability=True)
svm_model.fit(X_train, y_train)
print("Model loaded and ready for predictions!")

# Calculate and display model accuracy on test set
y_pred = svm_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1_score_value = f1_score(y_test, y_pred, pos_label='scream')  # Assuming 'scream' is the positive class

Loading saved training data...
Model loaded and ready for predictions!


In [3]:
def quick_predict(file_path):
    def extract_features(file_path):
        try:
            y, sr = librosa.load(file_path, sr=22050)
            features = {}
            
            # MFCC features
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            for i in range(13):
                features[f'mfcc{i+1}_mean'] = np.mean(mfcc[i]) 
                features[f'mfcc{i+1}_std'] = np.std(mfcc[i])   
            return features
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            return None
    
    features = extract_features(file_path)
    if features is None:
        print(f"Error processing {file_path}")
        return None
    
    feature_df = pd.DataFrame([features])
    
    for feature in feature_names:
        if feature not in feature_df.columns:
            feature_df[feature] = 0
    
    feature_df = feature_df[feature_names]
    
    features_scaled = scaler.transform(feature_df)
    
    prediction = svm_model.predict(features_scaled)[0]
    probabilities = svm_model.predict_proba(features_scaled)[0]
    
    result = {
        'file': file_path,
        'prediction': 'SCREAM' if prediction == 'scream' else 'NON-SCREAM',
        'confidence': max(probabilities),
        'scream_probability': probabilities[1] if prediction == 'scream' else probabilities[0]
    }
    
    # Print result with all metrics
    print(f"Confidence: {result['confidence']*100:.2f}%")
    print(f"Model Accuracy: {accuracy*100:.2f}%")
    print(f"Model F1 Score: {f1_score_value*100:.2f}%")
    print(f"Result: {result['prediction']}")

In [4]:
quick_predict("/kaggle/input/audio-dataset-of-scream-and-non-scream/Converted_Separately/scream/1014.wav")

Confidence: 99.34%
Model Accuracy: 95.37%
Model F1 Score: 95.60%
Result: SCREAM
